# Workshop: Large Language Models and the Impact of Tokenization

In this workshop, we will first explore how a _tokenizer_ works, the part of a language model that breaks up a text into smaller units.  Afterwards, we will try generating some text locally using a (very small) instruction fine-tuned LLM.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch
import gc

## Part A: Tokenization

In this part, we want to understand better how a tokenizer works, particularly in multilingual language models that were trained on hundreds of languages.  For this, we use the tokenizer of [mmBERT](https://huggingface.co/jhu-clsp/mmBERT-small) (“modern multilingual BERT”), an _encoder_ language model trained on 1800+ languages.  This model was only released in August 2025, and is a [modernized variant](https://huggingface.co/docs/transformers/main/en/model_doc/modernbert) of the classic BERT architecture.

We can download and instantiate the tokenizer from the Huggingface Hub as follows:

In [6]:
# This downloads & uses the model described at https://huggingface.co/jhu-clsp/mmBERT-small
bert_tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/mmBERT-small")

We can call the `tokenizer` object with any string of text:

In [7]:
inputs = bert_tokenizer("The capital of Sweden is Stockholm.").input_ids
inputs

[2, 714, 6037, 576, 24318, 603, 41106, 235265, 1]

This shows us what the text input “looks like” to a language model at the most basic level — a series of numbers.  It’s not very useful for _analyzing_ the tokenization though.  Luckily, it’s easy to convert these numbers (also called _IDs_) back to the actual _tokens_ they represent:

In [8]:
bert_tokenizer.convert_ids_to_tokens(inputs)

['<bos>',
 '▁The',
 '▁capital',
 '▁of',
 '▁Sweden',
 '▁is',
 '▁Stockholm',
 '.',
 '<eos>']

If we’re just interested in the tokenization, and not the IDs, we can even use the `.tokenize()` function to save us some work:

In [9]:
bert_tokenizer.tokenize("The capital of Sweden is Stockholm.")

['▁The', '▁capital', '▁of', '▁Sweden', '▁is', '▁Stockholm', '.']

Let’s try the same in Finnish:

In [10]:
bert_tokenizer.tokenize("Ruotsin pääkaupunki on Tukholma.")

['▁Ru',
 'ots',
 'in',
 '▁pää',
 'kau',
 'pun',
 'ki',
 '▁on',
 '▁Tuk',
 'hol',
 'ma',
 '.']

Try it out with your own examples!

### 📝 Task 1

1. **Compile your own list of at least _ten parallel sentence pairs_ in English and one other language of your choice.**
   - “Parallel” means that for each English sentence, the other sentence should be its direct translation in another language.  You can use your own language knowledge to produce these, or you can use a translation tool to help you.
   - Try to include a variety of sentences, some with simpler expressions, and some with more uncommon or technical words!
   - For the “other language”, you can pick _any_ language you like.
2. **Compute both the _average number of tokens_ and the _average fertility score_ for each language.**
   - Compute the number of tokens and fertility for each sentence, then average them separately for each language.
   - You can compute the _fertility_ by taking the _number of tokens_ and dividing it by the number of tokens that start with the special “beginning of word” marker `▁`.
     - _Tip:_ If your implementation is correct, the example sentence `"Ruotsin pääkaupunki on Tukholma."` should have a fertility of 3.
3. **Write a _brief_ reflection (max. 200 words) on the results that you got.**
   - Do the tokens in your chosen language correspond to “meaningful” parts of the words or do they seem arbitrary? Which language had a higher fertility on average, and what do you think this means?

In [11]:
# Additional sentences in English and Spanish
english_sentences = [
    "The capital of Sweden is Stockholm.",
    "I love reading books in the library.",
    "The weather today is very sunny.",
    "She studies computer science at university.",
    "My favorite food is pizza with extra cheese.",
    "The cat is sleeping on the sofa.",
    "We are going to the cinema tonight.",
    "The internet connection is very slow today.",
    "Artificial intelligence is transforming technology.",
    "The mountain range stretches across the continent."
]

spanish_sentences = [
    "La capital de Suecia es Estocolmo.",
    "Me encanta leer libros en la biblioteca.",
    "El clima hoy está muy soleado.",
    "Ella estudia informática en la universidad.",
    "Mi comida favorita es pizza con queso extra.",
    "El gato está durmiendo en el sofá.",
    "Vamos al cine esta noche.",
    "La conexión a internet está muy lenta hoy.",
    "La inteligencia artificial está transformando la tecnología.",
    "La cordillera se extiende por todo el continente."
]

In [12]:
# New Function to compute tokens and fertility for each sentence

def compute_fertility(tokens):
    """
    Compute fertility: number of tokens / number of tokens starting with '▁'
    """
    total_tokens = len(tokens)
    word_start_tokens = sum(1 for token in tokens if token.startswith('▁'))
    
    # Handle division by zero
    if word_start_tokens == 0:
        return 0.0
    
    fertility = total_tokens / word_start_tokens
    return fertility

In [13]:
# Additional code to tokenize English sentences

english_token_counts = []
english_fertilities = []

for sentence in english_sentences:
    tokens = bert_tokenizer.tokenize(sentence)
    token_count = len(tokens)
    fertility = compute_fertility(tokens)
    
    english_token_counts.append(token_count)
    english_fertilities.append(fertility)
    
    print(f"English: {sentence}")
    print(f"  Tokens: {tokens}")
    print(f"  Token count: {token_count}, Fertility: {fertility:.2f}")
    print()

English: The capital of Sweden is Stockholm.
  Tokens: ['▁The', '▁capital', '▁of', '▁Sweden', '▁is', '▁Stockholm', '.']
  Token count: 7, Fertility: 1.17

English: I love reading books in the library.
  Tokens: ['▁I', '▁love', '▁reading', '▁books', '▁in', '▁the', '▁library', '.']
  Token count: 8, Fertility: 1.14

English: The weather today is very sunny.
  Tokens: ['▁The', '▁weather', '▁today', '▁is', '▁very', '▁sunny', '.']
  Token count: 7, Fertility: 1.17

English: She studies computer science at university.
  Tokens: ['▁She', '▁studies', '▁computer', '▁science', '▁at', '▁university', '.']
  Token count: 7, Fertility: 1.17

English: My favorite food is pizza with extra cheese.
  Tokens: ['▁My', '▁favorite', '▁food', '▁is', '▁pizza', '▁with', '▁extra', '▁cheese', '.']
  Token count: 9, Fertility: 1.12

English: The cat is sleeping on the sofa.
  Tokens: ['▁The', '▁cat', '▁is', '▁sleeping', '▁on', '▁the', '▁sofa', '.']
  Token count: 8, Fertility: 1.14

English: We are going to the c

In [14]:
# Additional code to tokenize Spanish sentences
spanish_token_counts = []
spanish_fertilities = []

for sentence in spanish_sentences:
    tokens = bert_tokenizer.tokenize(sentence)
    token_count = len(tokens)
    fertility = compute_fertility(tokens)
    
    spanish_token_counts.append(token_count)
    spanish_fertilities.append(fertility)
    
    print(f"Spanish: {sentence}")
    print(f"  Tokens: {tokens}")
    print(f"  Token count: {token_count}, Fertility: {fertility:.2f}")
    print()

Spanish: La capital de Suecia es Estocolmo.
  Tokens: ['▁La', '▁capital', '▁de', '▁Suecia', '▁es', '▁Esto', 'col', 'mo', '.']
  Token count: 9, Fertility: 1.50

Spanish: Me encanta leer libros en la biblioteca.
  Tokens: ['▁Me', '▁encanta', '▁leer', '▁libros', '▁en', '▁la', '▁biblioteca', '.']
  Token count: 8, Fertility: 1.14

Spanish: El clima hoy está muy soleado.
  Tokens: ['▁El', '▁clima', '▁hoy', '▁está', '▁muy', '▁sole', 'ado', '.']
  Token count: 8, Fertility: 1.33

Spanish: Ella estudia informática en la universidad.
  Tokens: ['▁Ella', '▁estudia', '▁informática', '▁en', '▁la', '▁universidad', '.']
  Token count: 7, Fertility: 1.17

Spanish: Mi comida favorita es pizza con queso extra.
  Tokens: ['▁Mi', '▁comida', '▁favorita', '▁es', '▁pizza', '▁con', '▁queso', '▁extra', '.']
  Token count: 9, Fertility: 1.12

Spanish: El gato está durmiendo en el sofá.
  Tokens: ['▁El', '▁gato', '▁está', '▁dur', 'miendo', '▁en', '▁el', '▁sofá', '.']
  Token count: 9, Fertility: 1.29

Spanish:

In [15]:
# Compute averages
avg_english_tokens = sum(english_token_counts) / len(english_token_counts)
avg_english_fertility = sum(english_fertilities) / len(english_fertilities)

avg_spanish_tokens = sum(spanish_token_counts) / len(spanish_token_counts)
avg_spanish_fertility = sum(spanish_fertilities) / len(spanish_fertilities)

print("=" * 60)
print("SUMMARY:")
print(f"English - Average tokens: {avg_english_tokens:.2f}, Average fertility: {avg_english_fertility:.2f}")
print(f"Spanish - Average tokens: {avg_spanish_tokens:.2f}, Average fertility: {avg_spanish_fertility:.2f}")

SUMMARY:
English - Average tokens: 7.60, Average fertility: 1.15
Spanish - Average tokens: 8.50, Average fertility: 1.25


In [17]:
# Reflection on the above codes : 

reflection = """
REFLECTION:

The tokenization analysis reveals interesting patterns between English and Spanish. 
Spanish tokens generally correspond to meaningful morphological units—many splits 
align with morpheme boundaries. For example, "Estocolmo" splits as ['▁Esto', 'col', 'mo'], 
"soleado" as ['▁sole', 'ado'], and "transformando" as ['▁transform', 'ando'], where 
the splits often separate stems from affixes. Some splits appear less linguistically 
motivated, such as "cordillera" becoming ['▁cordi', 'll', 'era'].

Spanish achieved a higher average fertility of 1.25 compared to English's 1.15, 
meaning Spanish words require more tokens per word on average. This likely reflects 
Spanish's richer morphology (verb conjugations, compound words, inflections) and 
the tokenizer's tendency to split longer words into subword units. The higher average 
token count for Spanish (8.50 vs 7.60) further supports this.

The higher fertility suggests Spanish text is more "token-dense" than English for 
similar semantic content, which could impact model efficiency and representation 
learning in multilingual settings. The tokenizer appears to balance between 
linguistic awareness and compression efficiency.
"""

print(reflection)


REFLECTION:

The tokenization analysis reveals interesting patterns between English and Spanish. 
Spanish tokens generally correspond to meaningful morphological units—many splits 
align with morpheme boundaries. For example, "Estocolmo" splits as ['▁Esto', 'col', 'mo'], 
"soleado" as ['▁sole', 'ado'], and "transformando" as ['▁transform', 'ando'], where 
the splits often separate stems from affixes. Some splits appear less linguistically 
motivated, such as "cordillera" becoming ['▁cordi', 'll', 'era'].

Spanish achieved a higher average fertility of 1.25 compared to English's 1.15, 
meaning Spanish words require more tokens per word on average. This likely reflects 
Spanish's richer morphology (verb conjugations, compound words, inflections) and 
the tokenizer's tendency to split longer words into subword units. The higher average 
token count for Spanish (8.50 vs 7.60) further supports this.

The higher fertility suggests Spanish text is more "token-dense" than English for 
similar

## Part B: Using autoregressive LLMs

In this part, we want to experiment with generating text using _decoder_ LLMs.  Most recent LLMs, including commercial ones such as the models behind ChatGPT, Claude AI, or Copilot, are of this type, but there are also many openly-released models that you can download and run on your own computer.  Of course, many of these have quite significant hardware requirements (in terms of memory and/or GPU power), so we will only look at a very small model here that can be run on any modern laptop.

For this purpose, we use [**SmolLM2**](https://huggingface.co/HuggingFaceTB/SmolLM2-360M), a family of LLMs specifically trained to be compact in terms of number of parameters.  The downside is that these models were trained on primarily English text, and will probably not work well (or at all) in another language.

We start by loading the 360M-parameter, instruction-tuned version of SmolLM2, which has a size and memory footprint of about 1.5 GB:

In [18]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\punno\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\punno\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-360M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [19]:
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.2f} MB")

Memory footprint: 1447.28 MB


This model uses a slightly different tokenizer than mmBERT. Let’s see what it produces:

In [20]:
tokenizer.tokenize("The capital of Sweden is Stockholm.")

['The', 'Ġcapital', 'Ġof', 'ĠSweden', 'Ġis', 'ĠStockholm', '.']

Instead of the `▁` symbol, we now see the `Ġ` character to mark the beginning of words.  Also, the first word of the sentence does not have this marker.  This illustrates that there are different tokenizers and tokenizer implementations, so each model works a little bit differently in this regard, even though the general idea is the same!

Let’s now try generating some text with this model.  We can use a special class called `TextStreamer` to have the tokens printed to the notebook as they are being generated, just as in tools like ChatGPT:

In [21]:
streamer = TextStreamer(tokenizer)

Let’s try prompting it with a simple question:

In [22]:
inputs = tokenizer.encode("What is the capital of Sweden?", return_tensors="pt")
outputs = model.generate(inputs, streamer=streamer, max_new_tokens=50)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


What is the capital of Sweden?
A:<|im_end|>


Oops!  What happened here?  An instruction-tuned model is trained on input that follows a certain _template_, and we didn’t follow that when we wrote our prompt.  We can check the template within the tokenizer:

In [23]:
print(tokenizer.chat_template)

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


We can use the function `.apply_chat_template()` to turn a _list of messages_ into a prompt that’s formatted correctly according to the tokenizer’s chat template:

In [24]:
print(
    tokenizer.apply_chat_template([
        {"role": "user", "content": "What is the capital of Sweden?"},
    ], tokenize=False, add_generation_prompt=True)
)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of Sweden?<|im_end|>
<|im_start|>assistant



Let’s put it all together! We’ll use this chat template as the input to the model, and wrap it in a function to make it easier to use:

In [25]:
def generate(prompt, **kwargs):
    inputs = tokenizer.apply_chat_template([
        {"role": "user", "content": prompt},
    ], add_generation_prompt=True, return_tensors="pt")
    _ = model.generate(inputs, streamer=streamer, **kwargs)

generate("What is the capital of Sweden?", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of Sweden?<|im_end|>
<|im_start|>assistant
I'm sorry, but as a text-based AI, I don't have the ability to provide geographical information. I recommend using a mapping service like Google Maps or OpenStreetMap to find the capital of Sweden.<|im_end|>


Technically correct (maybe?), but maximally useless!

### 📝 Task 2

**Get the LLM to answer with “Stockholm” by only changing the prompt.**  Of course, do _not_ write the answer in the prompt, and do not use any of the techniques further down in the notebook!  Only change the wording of the question until the LLM responds with the correct answer.

Provide the code that you used for your generation below.

In [26]:
# Trying out different prompts
generate("What city is the capital of Sweden?", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What city is the capital of Sweden?<|im_end|>
<|im_start|>assistant
The capital of Sweden is Stockholm.<|im_end|>


In [27]:
generate("Name the capital city of Sweden.", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Name the capital city of Sweden.<|im_end|>
<|im_start|>assistant
The capital city of Sweden is Stockholm.<|im_end|>


In [28]:
generate("What is Sweden's capital city?", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is Sweden's capital city?<|im_end|>
<|im_start|>assistant
Sweden's capital city is Stockholm.<|im_end|>


In [29]:
generate("Which city is the capital of Sweden?", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Which city is the capital of Sweden?<|im_end|>
<|im_start|>assistant
Sweden's capital is Stockholm.<|im_end|>


In [30]:
generate("The capital of Sweden is what city?", max_new_tokens=50)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
The capital of Sweden is what city?<|im_end|>
<|im_start|>assistant
The capital of Sweden is Stockholm.<|im_end|>


### Sampling from a language model

When you run the generations we tried above multiple times, you will see that the LLM always generates the exact same response.  You might have noticed that a tool like ChatGPT doesn’t work that way, and will give you a slightly different text each time you run it.  This is because in practice, we don’t always want to pick the _most likely token_ to continue the text, but want to _sample_ from the probability distribution instead.

With our model, this can be achieved by setting `do_sample=True` and parameters like `top_k` to influence the “randomness” of the generations. If you run the following cell several times, you should get different outputs each time. Try it a few times!

In [31]:
generate("What is the capital of Sweden?", max_new_tokens=50, do_sample=True, top_k=10)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of Sweden?<|im_end|>
<|im_start|>assistant
I'd be happy to assist you in finding the capital of Sweden.<|im_end|>


Let’s try a completely different prompt now: asking how often the letter “r” appears in the word “strawberry”.  This is a [famous example](https://techcrunch.com/2024/08/27/why-ai-cant-spell-strawberry/) of a question that even big, commercial LLMs like GPT-4o and Claude struggle to get right.  Let’s see how our SmolLM2 model does:

In [32]:
generate("How many r's are in the word strawberry?", max_new_tokens=50, do_sample=False)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
How many r's are in the word strawberry?<|im_end|>
<|im_start|>assistant
There are 4 r's in the word strawberry.<|im_end|>


### 📝 Task 3

1. **Conduct a small experiment on how often the LLM correctly answers the question how many r’s are in “strawberry”.**  Try out the prompt with `do_sample=True`, try different values for `top_k`, and try reformulating the question if you wish.  Afterwards, decide on one setting and run the _same_ prompt with the _same_ parameters multiple times, but at least _five_ times, and keep track how often the LLM produced the correct answer.
2. **Write code to check how the LLM tokenizes your question.**  You can re-use the code from earlier in this notebook to do this.
3. **Write a _brief_ reflection (max. 200 words) on the following questions:**
   - How often did you run your prompt, and how often did the LLM give the correct answer?
   - Based on what you learned about tokenization, and how the LLM tokenizes your prompt, how do you explain this result?

In [ ]:
# YOUR CODE & REFLECTION CAN GO HERE

In [ ]:
# YOUR CODE & REFLECTION CAN GO HERE

In [ ]:
# YOUR CODE & REFLECTION CAN GO HERE